<a href="https://colab.research.google.com/github/kieron-s/Comp3610-Assignment3/blob/main/Comp3610_Assignment3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install pyspark -q

In [4]:
import time
import pandas as pd
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("COMP3610_Assignment3_NYC_Taxi")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("Warn")
print(f"Spark version: {spark.version}")
print(f"Application name: {spark.sparkContext.appName}")

Spark version: 4.0.2
Application name: COMP3610_Assignment3_NYC_Taxi


In [5]:
DATA_PATH = "/content/yellow_tripdata_2024-01.parquet"

spark_start = time.time()
df_raw = spark.read.parquet(DATA_PATH)
spark_row_count = df_raw.count()
spark_load_time = time.time() - spark_start

pandas_start = time.time()
df_pandas = pd.read_parquet(DATA_PATH)
pandas_load_time = time.time() - pandas_start

print(f"\nLoad Time Comparison")
print(f"Spark  load time : {spark_load_time:.2f}s  ({spark_row_count:,} rows)")
print(f"Pandas load time : {pandas_load_time:.2f}s  ({len(df_pandas):,} rows)")
print(f"\nInterpretation: Spark has higher overhead on a single machine due to JVM")
print(f"startup and task scheduling. However, Spark scales across clusters and handles")
print(f"datasets far exceeding available RAM, making it essential for big data workloads.")


Load Time Comparison
Spark  load time : 14.87s  (2,964,624 rows)
Pandas load time : 1.43s  (2,964,624 rows)

Interpretation: Spark has higher overhead on a single machine due to JVM
startup and task scheduling. However, Spark scales across clusters and handles
datasets far exceeding available RAM, making it essential for big data workloads.


In [6]:
print(f"DataFrame Schema")
df_raw.printSchema()
print(f"\nTotal Rows: {spark_row_count:,}")
print(f"\nPartition Information: {df_raw.rdd.getNumPartitions()}")

DataFrame Schema
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)


Total Rows: 2,964,624

Partition Information: 2


In [7]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

critical_cols = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "fare_amount",
    "trip_distance"
]

initial_count = df_raw.count()
print(f"Initial Row Count: {initial_count:,}")

df_no_nulls = df_raw.dropna(subset = critical_cols)
after_null_drop = df_no_nulls.count()
print(f"After Null Drop: {after_null_drop:,} (removed {initial_count - after_null_drop:,}")

df_filtered = df_no_nulls.filter(
    (F.col("trip_distance") > 0) &
    (F.col("fare_amount") >= 0) &
    (F.col("fare_amount") <= 500) &
    (F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime"))
)

after_filter = df_filtered.count()
print(f"After Filter: {after_filter:,} (removed {after_null_drop - after_filter:,}")
print(f"Total rows removed: {initial_count - after_filter:,}")

Initial Row Count: 2,964,624
After Null Drop: 2,964,624 (removed 0
After Filter: 2,870,046 (removed 94,578
Total rows removed: 94,578
